# Extract and plot scenario data for carbon scenarios

In [1]:
import pandas as pd
import gdxtools 
import glob
import os
import gams
import numpy as np
import gdxtools as gt
import plotly.express as px
import plotly

In [2]:
dir_in = "../../model/scenarios/carbon/results/evcost2000/"
fns = os.listdir(dir_in)
fns = list(filter(lambda k: 'gdx' in k, fns))

In [3]:
country_map = {'AT' : 'AUT', 'BE' : 'BEL', 'BG' : 'BGR', 'CH' : 'CHE', 'CZ' : 'CZE', 'DE' : 'DEU', 'DK' : 'DNK', 'ES' : 'ESP', 'FI' : 'FIN', 'FR' : 'FRA', 'GB' : 'GBR', 'GR' : 'GRC', 'HR' : 'HRV', 'HU' : 'HUN', 'IE' : 'IRL', 'IT' : 'ITA', 'LU' : 'LUX', 'NL' : 'NLD', 'NO' : 'NOR', 'PL' : 'POL', 'PT' : 'PRT', 'RO' : 'ROU', 'SE' : 'SWE', 'SI' : 'SVN', 'SK' : 'SVK'}

In [4]:
df_map_scenario_res_share=pd.DataFrame([
    ['carbonQCP_0.3_0.gdx',0.3,0],
    ['carbonQCP_0.4_0.gdx',0.4,0],
    ['carbonQCP_0.5_0.gdx',0.5,0],
    ['carbonQCP_0.6_0.gdx',0.6,0],
    ['carbonQCP_0.7_0.gdx',0.7,0],
    ['carbonQCP_0.8_0.gdx',0.8,0],
    ['carbonQCP_0.9_0.gdx',0.9,0],
    ['carbonQCP_1_0.gdx',1,0]
], columns=['scenario','res_share','ev_share'])

df_map_scenario_ev_share=pd.DataFrame([
    ['carbonQCP_0_0.1.gdx',0,0.1],
    ['carbonQCP_0_0.2.gdx',0,0.2],
    ['carbonQCP_0_0.3.gdx',0,0.3],
    ['carbonQCP_0_0.4.gdx',0,0.4],
    ['carbonQCP_0_0.5.gdx',0,0.5],
    ['carbonQCP_0_0.6.gdx',0,0.6],
    ['carbonQCP_0_0.7.gdx',0,0.7],
    ['carbonQCP_0_0.8.gdx',0,0.8],
    ['carbonQCP_0_0.9.gdx',0,0.9],
    ['carbonQCP_0_1.gdx',0,1],
], columns=['scenario','res_share','ev_share'])

df_map_scenario_base=pd.DataFrame([
    ['carbonQCP_0_0.gdx',0,0]
], columns=['scenario','res_share','ev_share'])    

In [5]:
dir_gms = os.getcwd()
ws = gams.GamsWorkspace(dir_gms)

In [6]:
# initialize dfs
df_welfare = pd.DataFrame()
df_modelstat = pd.DataFrame()

In [7]:
for fn in fns:    
    gdx = ws.add_database_from_gdx(dir_in+fn)
    df_welfare_temp = pd.DataFrame(gt.get_symbol_values(gdx, "r_welfare", col_names=["country"], kind="value")).reset_index()
    df_welfare_temp['scenario'] = fn
    df_welfare = df_welfare.append(df_welfare_temp)
    df_modelstat_temp = pd.DataFrame(gt.get_symbol_values(gdx, "r_modelstatistics", col_names=["stattype"], kind="value")).reset_index()
    df_modelstat_temp['scenario'] = fn
    df_modelstat = df_modelstat.append(df_modelstat_temp)
df_modelstat = df_modelstat.set_index('scenario')
df_welfare = df_welfare.set_index('scenario')

In [8]:
#rename countries to ISO3
df_welfare['country'] = df_welfare['country'].map(country_map)

In [9]:
#make pivot table
df_welfare_pivot = df_welfare.pivot_table(index ='scenario',columns='country',values='Value')

In [10]:
#exclude infeasible scenarios
df_modelstat = df_modelstat.pivot_table(index='scenario',columns='stattype',values='Value').reset_index()
df_modelstat = df_modelstat.rename_axis(None, axis=1)
df_modelstat = df_modelstat[['scenario','modelstat']]
df_modelstat = df_modelstat.set_index('scenario')
df_modelstat = df_modelstat[df_modelstat.modelstat == 1]
df_welfare_pivot = df_welfare_pivot.join(df_modelstat,how='inner')
df_welfare_pivot.drop('modelstat', axis=1, inplace=True)

In [11]:
#calculate relative change
df_welfare_pivot.loc[:,'AT':] = 100 * (1-df_welfare_pivot.loc[:,'AT':].div(df_welfare_pivot.loc['carbonQCP_0_0.gdx']['AT':]))
df_welfare_pivot.head()

,AUT,BEL,BGR,CHE,CZE,DEU,DNK,ESP,FIN,FRA,...,ITA,LUX,NLD,NOR,POL,PRT,ROU,SVK,SVN,SWE
scenario,,,,,,,,,,,,,,,,,,,,,
carbonQCP_0.3_0.gdx,9.340108e-07,6.551646e-07,8.227341e-09,8.903588e-07,6.590059e-07,6.996282e-07,8.103773e-09,1.219367e-07,-6.178413e-09,1.379418e-07,...,4.521401e-07,7.245393e-07,4.981440e-07,-6.041799e-07,0.000001,4.824940e-08,-2.492955e-07,8.203530e-07,4.594475e-07,-5.968703e-07
carbonQCP_0.4_0.gdx,1.489865e-06,1.088977e-06,-4.422551e-09,1.420259e-06,1.021389e-06,1.092446e-06,1.510225e-07,-2.693541e-08,1.339861e-07,1.869176e-07,...,7.201982e-07,1.129948e-06,7.879516e-07,-7.555084e-07,0.000002,-9.689805e-10,-1.072436e-06,1.198088e-06,6.508173e-07,-7.489048e-07
carbonQCP_0.5_0.gdx,-1.497626e-01,-9.708151e-02,-2.142312e-01,-1.325148e-01,-1.616783e-01,-1.547795e-01,-8.315118e-02,-1.154128e-01,-1.570342e-01,-1.515529e-01,...,-9.645669e-02,-1.571095e-01,-1.390672e-01,-4.836672e-02,-0.256758,-1.149798e-01,-1.684474e-01,-1.490507e-01,-1.273084e-01,-4.829811e-02
carbonQCP_0.6_0.gdx,-3.152202e+00,-2.138426e+00,-3.968095e+00,-2.788044e+00,-3.353930e+00,-2.914992e+00,-1.171020e+00,-2.663491e+00,-3.394081e+00,-2.933116e+00,...,-1.894489e+00,-2.944054e+00,-2.631259e+00,-1.584839e-01,-5.465685,-2.775564e+00,-3.506023e+00,-2.607660e+00,-3.130759e+00,1.023259e+00
carbonQCP_0.7_0.gdx,-3.412850e+00,-3.272213e+00,-3.369914e+00,-3.106578e+00,-3.919167e+00,-3.048961e+00,-1.229312e+00,-3.546577e+00,-3.146577e+00,-3.279695e+00,...,-2.419829e+00,-3.073110e+00,-2.797794e+00,2.046061e+00,-7.187860,-3.707862e+00,-3.343684e+00,-1.648418e+00,-3.325886e+00,2.078496e+00


In [12]:
# create separate dfs for res and ev share scenarios
df_welfare_res = df_welfare_pivot.merge(df_map_scenario_res_share, on='scenario').set_index(['scenario','res_share','ev_share'])
df_welfare_ev = df_welfare_pivot.merge(df_map_scenario_ev_share, on='scenario').set_index(['scenario','res_share','ev_share'])
df_welfare_res = pd.DataFrame(df_welfare_res.stack()).reset_index().rename(columns={0:'Value','level_3':'country'})
df_welfare_ev = pd.DataFrame(df_welfare_ev.stack()).reset_index().rename(columns={0:'Value','level_3':'country'})

In [20]:
fig = px.choropleth(df_welfare_res, locations = 'country',
              color = 'Value', range_color=[0,50],
              animation_frame = 'scenario',
              labels={"Value": "Welfare change [%]"},
              color_continuous_scale='PuRd',
              scope = 'europe', projection = 'equirectangular')
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_geos(
    lataxis_range=[34,65], lonaxis_range=[-10, 30],
    visible=False, showcountries=True, showland=True)
fig.write_html("../../figures/Welfare_change_res.html")
fig.show()

In [23]:
fig = px.choropleth(df_welfare_ev, locations = 'country',
              color = 'Value', range_color=[0,2],
              animation_frame = 'scenario',
              labels={"Value": "Welfare change [%]"},
              color_continuous_scale='PuRd',
              scope = 'europe', projection = 'equirectangular')
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_geos(
    lataxis_range=[34,65], lonaxis_range=[-10, 30],
    visible=False, showcountries=True, showland=True)
fig.write_html("../../figures/Welfare_change_ev.html")
fig.show()

In [15]:
df_welfare_sum = df_welfare.groupby('scenario').sum()

In [16]:
df_welfare_sum

,Value
scenario,
carbonQCP_0.3_0.gdx,-1.074697e+18
carbonQCP_0.4_0.gdx,-1.074697e+18
carbonQCP_0.5_0.gdx,-1.076151e+18
carbonQCP_0.6_0.gdx,-1.102606e+18
carbonQCP_0.7_0.gdx,-1.105464e+18
carbonQCP_0.8_0.gdx,-1.082979e+18
carbonQCP_0.9_0.gdx,-1.023399e+18
carbonQCP_0_0.1.gdx,-1.074697e+18
carbonQCP_0_0.2.gdx,-1.074697e+18


In [17]:
df_welfare_sum.to_csv("welfare.csv")